# WGBS Preprocessing — Step 1: Build Beta Matrices

**Goal:** Read per-sample Bismark ROI TSV files from 4 project directories and save clean CpG × sample beta-value matrices to `data/processed/`, ready for the ML pipeline.

| Dataset key | Source | Comparison | n (approx) 
|---|---|---|---|
| `stress` | nhip_cumulus | Control vs Stressed | ~70 |
| `wildfire` | macaque_nasal_HongJi | Control vs Exposed | ~22 |
| `obesity_hippocampus` | fromBen_MacaqueObese_brain / Hippocampus_OvC | Control vs Obese | ~13 |
| `obesity_hypothalamus` | fromBen_MacaqueObese_brain / Hypothalamus_OvC | Control vs Obese | ~13 |
| `obesity_prefrontalcortex` | fromBen_MacaqueObese_brain / PrefrontalCortex_OvC | Control vs Obese | ~13 |
| `cfdna_GD45` | fromBen_MacaqueObese_cfDna / GD45_OvC | Control vs Obese | ~12 |
| `cfdna_GD90` | fromBen_MacaqueObese_cfDna / GD90_OvC | Control vs Obese | ~12 |
| `cfdna_GD120` | fromBen_MacaqueObese_cfDna / GD120_OvC | Control vs Obese | ~12 |
| `cfdna_GD150` | fromBen_MacaqueObese_cfDna / GD150_OvC | Control vs Obese | ~12 |

**Outputs per dataset** (saved to `data/processed/`):
- `{dataset}_cpgi_methylation.csv` — CpGs × samples, beta values, CpG island window
- `{dataset}_genebody_methylation.csv` — CpGs × samples, beta values, gene body window
- `{dataset}_labels.csv` — sample_id, label (1=case, 0=control)

## Cell 1 — Imports

**What to remember:**
- `pathlib.Path` is the modern way to handle file paths in Python. It's cleaner than string concatenation and works on all operating systems.
- `glob.glob()` finds files matching a pattern (like `*.tsv.gz`). Think of it as a file search.
- `numpy` (as `np`) handles numerical arrays and math operations efficiently.
- `pandas` (as `pd`) handles tabular data — you'll use it constantly in bioinformatics.

In [1]:
import os
import glob
import numpy as np
import pandas as pd
from pathlib import Path

print('Libraries loaded.')

Libraries loaded.


## Cell 2 — Configuration

**What to remember:** All paths and parameters live here in one place — change them once, they propagate everywhere.

**Key change vs original pipeline:** We analyse the **full block** region `chr10:2,307,563-2,441,516` (~134 kb) as a **single unit** — no cpgi/genebody sub-window split. Every CpG in the region goes into one matrix per dataset, saved as `{dataset}_fullblock_methylation.csv`.

**Output directory:** `data/processed/block_analysis/` — completely separate from the original pipeline outputs so nothing is overwritten.


In [10]:
# ── Project root (one level up from this notebook's parent) ────────
PROJECT_ROOT = Path.cwd().parent.parent
# block_analysis notebooks live two levels below project root

# ── Output directory ────────────────────────────────────────────────
OUT_DIR = PROJECT_ROOT / 'data' / 'processed' / 'block_analysis'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Parent directory containing all dataset project folders ─────────
PARENT = Path('/quobyte/lasallegrp/Ensi/project')

# ── Genomic region of interest (full block, no sub-windows) ─────────
CHROM        = 'chr10'
REGION_START = 2307563
REGION_END   = 2441516
# Full block: chr10:2,307,563-2,441,516 (~134 kb)
# No cpgi/genebody sub-window split — entire region analysed as one unit

# ── Filtering parameters ─────────────────────────────────────────────
MIN_COV     = 1     # minimum read coverage per CpG per sample
MAX_MISSING = 0.7  # drop CpGs missing in >70% of samples

print(f'Output directory : {OUT_DIR}')
print(f'Region           : {CHROM}:{REGION_START:,}-{REGION_END:,} (~{(REGION_END-REGION_START)/1000:.0f} kb)')


Output directory : /quobyte/lasallegrp/Ensi/project/nhip_macaque/CpG_Methylation_ML/data/processed/block_analysis
Region           : chr10:2,307,563-2,441,516 (~134 kb)


## Cell 3 — Dataset Configuration

**What to remember:** This is a **dictionary of dictionaries** — a very common Python pattern for configuration. The outer key (e.g. `'stress'`) becomes the output file prefix. Each inner dict holds everything the processing function needs to know about that dataset.

**`parse_mode`** tells the parser how to extract the sample ID from the filename — each dataset has different naming conventions:
- `'roi1kb'` → stress + wildfire: `HJD036_merged_....roi1kb.tsv.gz` → `HJD036`
- `'brain'`  → obesity regions: `47383-Hippocampus_....roi.tsv.gz` → `47383-Hippocampus`
- `'cfdna'`  → cfDNA: `100224688.roi.tsv.gz` → `100224688`

**`meta_filter`** is used for cfDNA only — the metadata file covers all gestational days, so we filter it to get only the samples belonging to a specific GD timepoint.

In [14]:
DATASETS = {

    # ── Stress (NHIP preconception) ──────────────────────────────────
    'stress': {
        'roi_dir'     : PARENT / 'cumulus' / 'CpG_me2' / '08_cytosine_reports',
        'metadata'    : PARENT / 'nhip_macaque' / 'nhip_cumulus' / 'sample_info.csv',
        'meta_sep'    : '\t',
        'file_pattern': '*CpG_evidence.cov.gz.chr.txt.gz',
        'parse_mode'  : 'roi1kb',
        'ctrl_label'  : 'Control',
        'case_label'  : 'Stressed',
        'meta_filter' : None,           # no filtering needed
    },

    # ── Wildfire smoke (nasal epithelial) ────────────────────────────
    'wildfire': {
        'roi_dir'     : PARENT / 'nhip_macaque' / 'macaque_nasal_HongJi' / 'cpg_me2' / '06_methylation',
        'metadata'    : PARENT / 'nhip_macaque' / 'macaque_nasal_HongJi' / 'sample_info.csv',
        'meta_sep'    : ',',
        'file_pattern': '*.cov.gz',
        'parse_mode'  : 'roi1kb',
        'ctrl_label'  : 'Control',
        'case_label'  : 'Exposed',
        'meta_filter' : None,
    },

    # ── Obesity — Hippocampus ─────────────────────────────────────────
    'obesity_hippocampus': {
        'roi_dir'     : PARENT / 'nhip_macaque' / 'fromBen_MacaqueObese_brain' / 'DMRs' / 'Hippocampus_OvC',
        'metadata'    : PARENT / 'nhip_macaque' / 'fromBen_MacaqueObese_brain' / 'DMRs' / 'sample_info_master.csv',
        'meta_sep'    : '\t',
        'file_pattern': '*_evidence.cov.gz',
        'parse_mode'  : 'brain',
        'ctrl_label'  : 'Control',
        'case_label'  : 'Obese',
        'meta_filter' : None,
        # Brain region is already encoded in the Name (e.g. 47383-Hippocampus)
        # so filenames and metadata match naturally without extra filtering
    },

    # ── Obesity — Hypothalamus ────────────────────────────────────────
    'obesity_hypothalamus': {
        'roi_dir'     : PARENT / 'nhip_macaque' / 'fromBen_MacaqueObese_brain' / 'DMRs' / 'Hypothalamus_OvC',
        'metadata'    : PARENT / 'nhip_macaque' / 'fromBen_MacaqueObese_brain' / 'DMRs' / 'sample_info_master.csv',
        'meta_sep'    : '\t',
        'file_pattern': '*_evidence.cov.gz',
        'parse_mode'  : 'brain',
        'ctrl_label'  : 'Control',
        'case_label'  : 'Obese',
        'meta_filter' : None,
    },

    # ── Obesity — Prefrontal Cortex ───────────────────────────────────
    'obesity_prefrontalcortex': {
        'roi_dir'     : PARENT / 'nhip_macaque' / 'fromBen_MacaqueObese_brain' / 'DMRs' / 'PrefrontalCortex_OvC',
        'metadata'    : PARENT / 'nhip_macaque' / 'fromBen_MacaqueObese_brain' / 'DMRs' / 'sample_info_master.csv',
        'meta_sep'    : '\t',
        'file_pattern': '*_evidence.cov.gz',
        'parse_mode'  : 'brain',
        'ctrl_label'  : 'Control',
        'case_label'  : 'Obese',
        'meta_filter' : None,
    },
    # ── cfDNA — Gestational Day 45 ───────────────────────────────────
    'cfdna_GD45': {
        'roi_dir'     : PARENT / 'nhip_macaque' / 'fromBen_MacaqueObese_cfDna' / 'DMRs' / 'GD45_OvC',
        'metadata'    : PARENT / 'nhip_macaque' / 'fromBen_MacaqueObese_cfDna' / 'DMRs' / 'master_sample_info_cfDNA.csv',
        'meta_sep'    : ',',
        'file_pattern': '*.roiBlock.tsv.gz',
        'parse_mode'  : 'cfdna',
        'ctrl_label'  : 'Control',
        'case_label'  : 'Obese',
        'meta_filter' : ('Folder', 'GD45_OvC'),
    },
    
    # ── cfDNA — Gestational Day 90 ────────────────────────────────────
    'cfdna_GD90': {
        'roi_dir'     : PARENT / 'nhip_macaque' / 'fromBen_MacaqueObese_cfDna' / 'DMRs' / 'GD90_OvC',
        'metadata'    : PARENT / 'nhip_macaque' / 'fromBen_MacaqueObese_cfDna' / 'DMRs' / 'master_sample_info_cfDNA.csv',
        'meta_sep'    : ',',
        'file_pattern': '*.roiBlock.tsv.gz',
        'parse_mode'  : 'cfdna',
        'ctrl_label'  : 'Control',
        'case_label'  : 'Obese',
        'meta_filter' : ('Folder', 'GD90_OvC'),
        # meta_filter = (column_name, value) — filters metadata to this GD only
        # The cfDNA metadata covers all GDs in one file, so we must filter
    },

    # ── cfDNA — Gestational Day 120 ───────────────────────────────────
    'cfdna_GD120': {
        'roi_dir'     : PARENT / 'nhip_macaque' / 'fromBen_MacaqueObese_cfDna' / 'DMRs' / 'GD120_OvC',
        'metadata'    : PARENT / 'nhip_macaque' / 'fromBen_MacaqueObese_cfDna' / 'DMRs' / 'master_sample_info_cfDNA.csv',
        'meta_sep'    : ',',
        'file_pattern': '*.roiBlock.tsv.gz',
        'parse_mode'  : 'cfdna',
        'ctrl_label'  : 'Control',
        'case_label'  : 'Obese',
        'meta_filter' : ('Folder', 'GD120_OvC'),
    },

    # ── cfDNA — Gestational Day 150 ───────────────────────────────────
    'cfdna_GD150': {
        'roi_dir'     : PARENT / 'nhip_macaque' / 'fromBen_MacaqueObese_cfDna' / 'DMRs' / 'GD150_OvC',
        'metadata'    : PARENT / 'nhip_macaque' / 'fromBen_MacaqueObese_cfDna' / 'DMRs' / 'master_sample_info_cfDNA.csv',
        'meta_sep'    : ',',
        'file_pattern': '*.roiBlock.tsv.gz',
        'parse_mode'  : 'cfdna',
        'ctrl_label'  : 'Control',
        'case_label'  : 'Obese',
        'meta_filter' : ('Folder', 'GD150_OvC'),
    },
}

print(f'{len(DATASETS)} datasets configured:')
for name in DATASETS:
    print(f'  {name}')

9 datasets configured:
  stress
  wildfire
  obesity_hippocampus
  obesity_hypothalamus
  obesity_prefrontalcortex
  cfdna_GD45
  cfdna_GD90
  cfdna_GD120
  cfdna_GD150


## Cell 4 — Helper Functions

### `parse_sample_name()` — Why three modes?
Different labs name files differently. One function with a `mode` argument handles all cases cleanly.

### `load_sample()` — Key concepts:
- Reads a Bismark TSV, filters to `CHROM:REGION_START-REGION_END`
- Computes beta = meth / (meth + unmeth), marks low-coverage CpGs as NaN

### `build_matrix()` — Core alignment trick:
Each sample becomes a `pd.Series(index=CpG_positions, values=beta)`.
`pd.DataFrame(dict_of_series)` aligns all series automatically — missing positions become NaN.

### `filter_missing()` — Quality control:
Drops CpGs that are missing in more than `MAX_MISSING` fraction of samples.

**Note:** `subset_window()` from the original pipeline is removed here — we keep the entire
chr10:2,307,563-2,441,516 block as one unit with no sub-window splitting.


In [12]:
# ── 1. Sample name parser ────────────────────────────────────────────

def parse_sample_name(fpath: str, mode: str) -> str:
    """
    Extract sample ID from filename. Three modes for three naming conventions.

    roi1kb : HJD036_merged_....roi1kb.tsv.gz     -> HJD036
    brain  : 47383-Hippocampus_ZR3377_....roi.tsv.gz -> 47383-Hippocampus
    cfdna  : 100224688.roi.tsv.gz                -> 100224688
    """
    base = os.path.basename(fpath)

    if mode == 'roi1kb':
        base = base.replace('.roi1kb.tsv.gz', '')
        return base.split('_', 1)[0]

    elif mode == 'brain':
        base = base.replace('.roi.tsv.gz', '')
        return base.split('_', 1)[0]

    elif mode == 'cfdna':
        return base.replace('.roi.tsv.gz', '')

    else:
        raise ValueError(f'Unknown parse_mode: {mode}')


# ── 2. Load one sample ───────────────────────────────────────────────

def load_sample(fpath: str) -> pd.DataFrame:
    """
    Read one Bismark ROI TSV file. Returns a DataFrame with columns:
      start : CpG genomic position (used as the row identifier)
      beta  : methylation level 0-1 (= meth_reads / total_reads)
    """
    cols = ['chrom', 'start', 'end', 'pct', 'meth', 'unmeth']

    df = pd.read_csv(
        fpath,
        sep='\t',
        header=None,
        names=cols,
        compression='gzip'
    )

    if df.empty:
        return df

    df = df[df['chrom'] == CHROM].copy()
    if df.empty:
        return df

    # Filter to the full block region
    df = df[
        (df['start'] >= REGION_START) &
        (df['start'] <= REGION_END)
    ].copy()
    if df.empty:
        return df

    df['cov']  = df['meth'].astype(float) + df['unmeth'].astype(float)
    df['beta'] = np.where(
        df['cov'] > 0,
        df['meth'].astype(float) / df['cov'],
        np.nan
    )

    df = df[df['cov'] >= MIN_COV].copy()
    df = df.dropna(subset=['beta'])

    return df[['start', 'beta']].reset_index(drop=True)


# ── 3. Build CpG x samples matrix ───────────────────────────────────

def build_matrix(files, name_to_group, ctrl_label, case_label, parse_mode):
    """
    Load all sample files and combine into one CpG x samples DataFrame.
    Missing positions become NaN and are handled downstream by filter_missing + imputation.
    """
    sample_series = {}
    labels        = {}

    for fpath in files:
        name  = parse_sample_name(fpath, parse_mode)
        group = name_to_group.get(name)

        if group is None:
            print(f'  [SKIP] {name} -- not in metadata')
            continue
        if group not in (ctrl_label, case_label):
            print(f'  [SKIP] {name} -- group "{group}" not in comparison')
            continue

        df = load_sample(fpath)
        if df.empty:
            print(f'  [SKIP] {name} -- no data in region')
            continue

        s = df.set_index('start')['beta']
        s.name = name
        sample_series[name] = s
        labels[name] = 1 if group == case_label else 0

    if not sample_series:
        raise RuntimeError('No samples loaded. Check paths and metadata Name column.')

    beta_matrix = pd.DataFrame(sample_series)
    beta_matrix.index.name = 'CpG_start'

    print(f'  Loaded: {beta_matrix.shape[1]} samples x {beta_matrix.shape[0]} CpGs (before filtering)')
    return beta_matrix, labels


# ── 4. Filter CpGs with too much missing data ────────────────────────

def filter_missing(beta_matrix):
    """
    Drop CpGs missing in more than MAX_MISSING fraction of samples.
    """
    n        = beta_matrix.shape[1]
    frac_na  = beta_matrix.isna().sum(axis=1) / n
    kept     = beta_matrix[frac_na <= MAX_MISSING].copy()
    dropped  = beta_matrix.shape[0] - kept.shape[0]
    print(f'  Missing filter: dropped {dropped} CpGs -> {kept.shape[0]} retained')
    return kept


# subset_window() removed — block analysis uses the full region as one unit.

print('Helper functions defined.')


Helper functions defined.


## Cell 5 — Main Processing Loop

**What to remember:** `for name, cfg in DATASETS.items()` is how you loop over a dictionary getting both the key and value simultaneously. `.items()` returns (key, value) pairs — you'll use this constantly.

**`glob.glob()`** finds all files matching a pattern. The `str()` call is needed because glob expects a string, not a Path object.

**`dict(zip(meta['Name'], meta['Group']))`** — this is a very common bioinformatics one-liner: `zip()` pairs two lists together, `dict()` converts those pairs into a fast lookup table. Result: `{'HJD036': 'Control', 'HJD037': 'Exposed', ...}`

In [13]:
summary_rows = []   # collect per-dataset results for the final summary table

for ds_name, cfg in DATASETS.items():
    print(f"\n{'='*60}")
    print(f"Dataset: {ds_name}")
    print(f"{'='*60}")

    # ── Find sample files ────────────────────────────────────────────
    files = sorted(glob.glob(str(cfg['roi_dir'] / cfg['file_pattern'])))
    if not files:
        print(f'  [ERROR] No files found in {cfg["roi_dir"]} — skipping.')
        continue
    print(f'  Files found: {len(files)}')

    # ── Load metadata ────────────────────────────────────────────────
    meta = pd.read_csv(cfg['metadata'], sep=cfg['meta_sep'])
    meta['Name']  = meta['Name'].astype(str)
    meta['Group'] = meta['Group'].astype(str)

    if cfg['meta_filter'] is not None:
        col, val = cfg['meta_filter']
        meta = meta[meta[col] == val].copy()
        print(f'  Metadata filtered to {col}={val}: {len(meta)} samples')

    name_to_group = dict(zip(meta['Name'], meta['Group']))
    ctrl = cfg['ctrl_label']
    case = cfg['case_label']
    n_ctrl = sum(1 for g in name_to_group.values() if g == ctrl)
    n_case = sum(1 for g in name_to_group.values() if g == case)
    print(f'  Metadata: {n_ctrl} {ctrl} / {n_case} {case}')

    # ── Build beta matrix ────────────────────────────────────────────
    try:
        beta_matrix, labels = build_matrix(
            files, name_to_group, ctrl, case, cfg['parse_mode']
        )
    except RuntimeError as e:
        print(f'  [ERROR] {e}')
        continue

    # ── Filter missing CpGs ──────────────────────────────────────────
    beta_matrix = filter_missing(beta_matrix)

    # ── Impute remaining NaNs with per-CpG mean ──────────────────────
    # axis=1 means we operate row by row (per CpG), filling NaN with
    # that CpG's mean beta across all samples.
    beta_matrix = beta_matrix.apply(lambda row: row.fillna(row.mean()), axis=1)
    n_imputed = beta_matrix.isna().sum().sum()
    print(f'  After imputation: {n_imputed} NaNs remaining (should be 0)')

    # ── No sub-window split — save the full block directly ───────────
    # The entire chr10:2,307,563-2,441,516 region is one analysis unit.
    # CpGs are already filtered to REGION_START..REGION_END by load_sample().
    print(f'  Full block CpGs retained: {beta_matrix.shape[0]}')

    # ── Save output files ────────────────────────────────────────────
    block_path = OUT_DIR / f'{ds_name}_fullblock_methylation.csv'
    label_path = OUT_DIR / f'{ds_name}_labels.csv'

    beta_matrix.to_csv(block_path)
    labels_df = pd.DataFrame(list(labels.items()), columns=['sample_id', 'label'])
    labels_df.to_csv(label_path, index=False)

    print(f'  Saved: {block_path.name}')
    print(f'         {label_path.name}')

    summary_rows.append({
        'dataset'       : ds_name,
        'n_samples'     : len(labels),
        'n_ctrl'        : sum(1 for v in labels.values() if v == 0),
        'n_case'        : sum(1 for v in labels.values() if v == 1),
        'fullblock_cpgs': beta_matrix.shape[0],
    })

print(f"\n{'='*60}")
print('All datasets processed.')
print(f"{'='*60}")



Dataset: stress
  Files found: 78
  Metadata: 52 Control / 25 Stressed
  [SKIP] CF58 -- not in metadata
  [SKIP] MC04 -- no data in region


KeyboardInterrupt: 

## Cell 6 — Summary Table

Check this table after processing to confirm:
- All 9 datasets loaded with the expected sample counts
- Class balance looks reasonable
- `fullblock_cpgs` shows how many CpGs survived filtering across the ~134 kb window

A larger region means more CpGs — expect hundreds to low thousands depending on WGBS coverage.


In [ ]:
summary = pd.DataFrame(summary_rows)
summary = summary.set_index('dataset')
print('\n── Preprocessing Summary ──')
print(summary.to_string())

# Save summary to results/tables/
tables_dir = PROJECT_ROOT / 'results' / 'tables'
tables_dir.mkdir(parents=True, exist_ok=True)
summary.to_csv(tables_dir / 'preprocessing_summary.csv')
print(f'\nSummary saved to results/tables/preprocessing_summary.csv')

## Cell 7 — Spot Check

Sanity-check the processed data before running ML:
- Beta values must be between 0 and 1
- Sample IDs should match your metadata
- CpG positions should all fall within `REGION_START`..`REGION_END`

`df.head()` shows the first 5 rows. `df.describe()` gives min/max/mean.


In [ ]:
# Spot-check one dataset — change 'stress' to any dataset key
check_ds = 'stress'

block_check = pd.read_csv(OUT_DIR / f'{check_ds}_fullblock_methylation.csv', index_col=0)
print(f'=== {check_ds} — Full block matrix ===')
print(f'Shape: {block_check.shape[0]} CpGs x {block_check.shape[1]} samples')
print(f'\nFirst 5 rows:')
display(block_check.head())

print(f'\nBeta value range (should be 0-1):')
print(f'  Min:  {block_check.values.min():.4f}')
print(f'  Max:  {block_check.values.max():.4f}')
print(f'  Mean: {block_check.values.mean():.4f}')

labels_check = pd.read_csv(OUT_DIR / f'{check_ds}_labels.csv')
print(f'\nLabels (first 5):')
display(labels_check.head())
